In [4]:
# importing packages
from pytubefix import YouTube
import os

# url input from youtube
yt = YouTube(r"https://www.youtube.com/watch?v=zriFfGkQ2dY")

# extract only audio
video = yt.streams.filter(only_audio=True).first()

# set destination to save file
destination = (r"C:\Users\Edopi\Desktop\2024-25c-fai2-adsai-EdoardoPierezza231412\Evidence\Task2")

# download the file
out_file = video.download(output_path=destination)

# save the file
base, ext = os.path.splitext(out_file)
new_file = base + '.mp3'
os.rename(out_file, new_file)

In [2]:
import requests
import time
import re

# Replace with your actual AssemblyAI API key.
ASSEMBLYAI_API_KEY = "1171e036276f45e1970fbefd5d8bfaec"
TRANSCRIPTION_ENDPOINT = "https://api.assemblyai.com/v2/transcript"
UPLOAD_ENDPOINT = "https://api.assemblyai.com/v2/upload"

def upload_file(file_path):
    """
    Uploads a local file (e.g., an MP3) to AssemblyAI and returns the file's URL.
    """
    headers = {"authorization": ASSEMBLYAI_API_KEY}
    with open(file_path, 'rb') as f:
        response = requests.post(UPLOAD_ENDPOINT, headers=headers, data=f)
    response.raise_for_status()
    upload_url = response.json()['upload_url']
    return upload_url

def transcribe_audio(audio_url):
    """
    Submits the uploaded audio URL to AssemblyAI for transcription.
    """
    headers = {
        "authorization": ASSEMBLYAI_API_KEY,
        "content-type": "application/json"
    }
    data = {
        "audio_url": audio_url,
        "language_code": "en_us"  # Adjust language if needed.
    }
    response = requests.post(TRANSCRIPTION_ENDPOINT, json=data, headers=headers)
    response.raise_for_status()
    transcript_id = response.json()['id']
    return transcript_id

def poll_transcription(transcript_id):
    """
    Polls AssemblyAI until the transcription is complete.
    Returns the full transcript text when done.
    """
    headers = {"authorization": ASSEMBLYAI_API_KEY}
    polling_url = f"{TRANSCRIPTION_ENDPOINT}/{transcript_id}"
    
    while True:
        response = requests.get(polling_url, headers=headers)
        response.raise_for_status()
        status = response.json()['status']
        if status == 'completed':
            return response.json()['text']
        elif status == 'error':
            raise Exception("Transcription failed: " + response.json().get('error', 'Unknown error'))
        print(f"Transcription status: {status}. Waiting for 5 seconds...")
        time.sleep(5)

def split_into_sentences(text):
    """
    Splits the transcript text into sentences.
    The regex looks for punctuation (. ! or ?) followed by whitespace.
    """
    sentences = re.split(r'(?<=[.!?])\s+', text)
    return sentences

def write_sentences_to_file(sentences, output_file):
    """
    Writes each sentence to the output file, one sentence per line.
    """
    with open(output_file, 'w', encoding='utf-8') as f:
        for sentence in sentences:
            f.write(sentence.strip() + "\n")

if __name__ == '__main__':
    # Replace with the path to your MP3 file.
    mp3_file_path = r"C:\Users\Edopi\Desktop\2024-25c-fai2-adsai-group-team_9\Data_lab_task\Task 2\Survivor NZ  Season 1 (2016)  Episode 1 - FULL EPISODE.mp3"
    
    try:
        print("Uploading file...")
        audio_url = upload_file(mp3_file_path)
        print("File uploaded successfully. Audio URL:", audio_url)
        
        print("Starting transcription...")
        transcript_id = transcribe_audio(audio_url)
        print("Transcription job started with ID:", transcript_id)
        
        transcript_text = poll_transcription(transcript_id)
        print("Transcription completed successfully.")
        
        sentences = split_into_sentences(transcript_text)
        output_file = "output.txt"
        write_sentences_to_file(sentences, output_file)
        print(f"Sentences written to {output_file}")
        
    except Exception as e:
        print("An error occurred:", e)



Uploading file...
File uploaded successfully. Audio URL: https://cdn.assemblyai.com/upload/1841f6f1-e72a-49e3-8679-fe06bcd271df
Starting transcription...
Transcription job started with ID: e7e9c7d4-96c9-4412-aa33-dc8b745e558e
Transcription status: processing. Waiting for 5 seconds...
Transcription status: processing. Waiting for 5 seconds...
Transcription status: processing. Waiting for 5 seconds...
Transcription status: processing. Waiting for 5 seconds...
Transcription status: processing. Waiting for 5 seconds...
Transcription completed successfully.
Sentences written to output.txt


## Bronze medal

In [7]:
import requests
import time
import re
import csv

ASSEMBLYAI_API_KEY = "1171e036276f45e1970fbefd5d8bfaec"
TRANSCRIPTION_ENDPOINT = "https://api.assemblyai.com/v2/transcript"
UPLOAD_ENDPOINT = "https://api.assemblyai.com/v2/upload"

def upload_file(file_path):
    """Uploads a local file to AssemblyAI and returns the upload URL"""
    headers = {"authorization": ASSEMBLYAI_API_KEY}
    with open(file_path, 'rb') as f:
        response = requests.post(UPLOAD_ENDPOINT, headers=headers, data=f)
    response.raise_for_status()
    return response.json()['upload_url']

def transcribe_audio(audio_url):
    """Starts transcription of the audio file"""
    headers = {
        "authorization": ASSEMBLYAI_API_KEY,
        "content-type": "application/json"
    }
    data = {"audio_url": audio_url, "language_code": "en_us"}
    response = requests.post(TRANSCRIPTION_ENDPOINT, json=data, headers=headers)
    response.raise_for_status()
    return response.json()['id']

def poll_transcription(transcript_id):
    """Polls AssemblyAI until transcription is complete"""
    headers = {"authorization": ASSEMBLYAI_API_KEY}
    polling_url = f"{TRANSCRIPTION_ENDPOINT}/{transcript_id}"
    
    while True:
        response = requests.get(polling_url, headers=headers)
        response.raise_for_status()
        status = response.json()['status']
        if status == 'completed':
            return response.json()['text']
        elif status == 'error':
            raise Exception("Transcription failed: " + response.json().get('error', 'Unknown error'))
        time.sleep(5)

def split_into_sentences(text):
    """Splits text into sentences using punctuation markers"""
    return re.split(r'(?<=[.!?])\s+', text)

def correct_transcript(text):
    """Applies regex corrections to the text and returns corrected text with correction count"""
    corrections = [
        (r'\bgrammer\b', 'grammar'),
        (r'\bteh\b', 'the'),
        (r'\bther\b', 'there'),
        (r'\bNew Zeeland\b', 'New Zealand'),
        (r'\bsoftwere\b', 'software'),
        (r'\bwanna\b', 'want to'),
        (r'\bgonna\b', 'going to'),
        (r'\bcause\b', 'because'),
        (r'\bwhats\b', "what's"),
        (r'\bim\b', "I'm"),
        # Tribe name fixes
        (r'\bBogaton\b', 'Mogotón'),
        (r'\bMimosa\b', 'Hermosa'),
        (r'\bYohumosa\b', 'Hermosa'),
        
        # Common misheard words
        (r'\bfelon\b', 'villain'),
        (r'\bweekly\b', 'weakly'),
        (r'\bmummy\b', 'mommy'),
        (r'\bArby\'?s\b', "Barb's"),
        
        # Capitalization fixes
        (r'\bStuff\b', 'stuff'),
        (r'\bRedemption Islands\b', 'Redemption Island'),
        
        # Missing words
        (r'\ball I need water\b', 'all I need is water'),
        (r'\bunit tractor track\b', 'unit tracker track'),
        
        # Common contractions
        (r'\bcoulda\b', 'could have'),
        (r'\bwoulda\b', 'would have'),
        (r'\bshoulda\b', 'should have'),
        
        # Survivor-specific terms
        (r'\bTribal Counsel\b', 'Tribal Council'),
        (r'\bImmunity Idol\b', 'Immunity Idol'),
        
        # Grammatical improvements
        (r'\baint\b', "isn't"),
        (r'\byoure\b', "you're"),
        (r'\btheyre\b', "they're")
    ]
    
    total_corrections = 0
    for pattern, replacement in corrections:
        text, n = re.subn(pattern, replacement, text, flags=re.IGNORECASE)
        total_corrections += n
    return text, total_corrections

def write_to_csv(original_sentences, corrected_sentences, output_file):
    """Writes comparison CSV with original and corrected transcripts"""
    with open(output_file, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(["Original Transcription", "Corrected Transcription"])
        for orig, corr in zip(original_sentences, corrected_sentences):
            writer.writerow([orig.strip(), corr.strip()])

if __name__ == '__main__':
    mp3_file_path = r"C:\Users\Edopi\Desktop\2024-25c-fai2-adsai-group-team_9\Data_lab_task\Task 2\Survivor NZ  Season 1 (2016)  Episode 1 - FULL EPISODE.mp3"
    
    try:
        print("Uploading file...")
        audio_url = upload_file(mp3_file_path)
        
        print("Starting transcription...")
        transcript_id = transcribe_audio(audio_url)
        
        transcript_text = poll_transcription(transcript_id)
        print("Transcription completed.")
        
        # Process sentences and track corrections
        original_sentences = split_into_sentences(transcript_text)
        corrected_sentences = []
        total_corrections = 0
        
        for sentence in original_sentences:
            corrected_sentence, corrections = correct_transcript(sentence)
            corrected_sentences.append(corrected_sentence)
            total_corrections += corrections
        
        # Generate CSV output
        output_file = "transcript_comparison.csv"
        write_to_csv(original_sentences, corrected_sentences, output_file)
        
        print(f"\nSummary Report:")
        print(f"Total sentences processed: {len(original_sentences)}")
        print(f"Total corrections made: {total_corrections}")
        print(f"CSV report generated: {output_file}")
        
    except Exception as e:
        print(f"Error: {str(e)}")

Uploading file...
Starting transcription...
Transcription completed.

Summary Report:
Total sentences processed: 1058
Total corrections made: 36
CSV report generated: transcript_comparison.csv
